# Diffusion 실습 — MNIST로 DDPM 처음부터 학습하기

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/diffusion_ddpm_mnist.ipynb)

**먼저:** 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` 를 고른 뒤 위에서부터 차례로 실행하세요.

1. 앞방향(forward): 깨끗한 이미지에 노이즈를 조금씩 섞어 완전한 노이즈로 만든다.
2. 학습: 작은 UNet이 "섞인 노이즈 $\epsilon$"을 맞히도록 학습한다. 손실 $\|\epsilon - \epsilon_\theta(x_t, t)\|^2$
3. 뒷방향(reverse): 순수 노이즈에서 시작해 예측한 노이즈를 조금씩 빼면 숫자가 나타난다.

T4 GPU 기준 학습은 약 5~10분이 걸립니다.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, math
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 1. 데이터: MNIST를 [-1, 1] 범위로

In [ ]:
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
ds = datasets.MNIST('data', train=True, download=True, transform=tf)
loader = DataLoader(ds, batch_size=128, shuffle=True, num_workers=2, drop_last=True)
x0, _ = next(iter(loader))
print(x0.shape, x0.min().item(), x0.max().item())

## 2. 노이즈 스케줄과 앞방향 과정

$\beta_t$ 를 1e-4 에서 0.02 까지 선형으로 늘리고 $\bar\alpha_t = \prod_{s\le t}(1-\beta_s)$ 라 두면, 한 번에

$$x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon,\quad \epsilon\sim\mathrm{N}(0, I)$$

로 아무 단계 $t$ 의 노이즈 섞인 이미지를 바로 만들 수 있습니다.

In [ ]:
T = 1000
betas = torch.linspace(1e-4, 0.02, T, device=device)
alphas = 1 - betas
alpha_bar = torch.cumprod(alphas, dim=0)

def q_sample(x0, t, noise):
    ab = alpha_bar[t].view(-1, 1, 1, 1)
    return ab.sqrt() * x0 + (1 - ab).sqrt() * noise

# 한 장에 노이즈가 섞여 가는 모습
img = x0[:1].to(device)
steps = [0, 50, 100, 200, 400, 600, 800, 999]
fig, axes = plt.subplots(1, len(steps), figsize=(14, 2))
for ax, t in zip(axes, steps):
    xt = q_sample(img, torch.tensor([t], device=device), torch.randn_like(img))
    ax.imshow(xt[0, 0].cpu(), cmap='gray', vmin=-1, vmax=1); ax.set_title(f't={t}'); ax.axis('off')
plt.show()

## 3. 노이즈 예측 모델: 작은 UNet

입력은 노이즈 섞인 이미지 $x_t$ 와 단계 $t$ 이고, 출력은 같은 크기의 노이즈 예측 $\epsilon_\theta(x_t, t)$ 입니다.
$t$ 는 Transformer 위치 인코딩과 같은 sin/cos 임베딩으로 넣습니다.

In [ ]:
def time_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
    args = t.float()[:, None] * freqs[None]
    return torch.cat([args.sin(), args.cos()], dim=1)

class Block(nn.Module):
    def __init__(self, cin, cout, tdim):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, padding=1)
        self.conv2 = nn.Conv2d(cout, cout, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, cout)
        self.norm2 = nn.GroupNorm(8, cout)
        self.tproj = nn.Linear(tdim, cout)
        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()
    def forward(self, x, temb):
        h = F.silu(self.norm1(self.conv1(x)))
        h = h + self.tproj(temb)[:, :, None, None]
        h = F.silu(self.norm2(self.conv2(h)))
        return h + self.skip(x)

class UNet(nn.Module):
    def __init__(self, ch=64, tdim=128):
        super().__init__()
        self.tdim = tdim
        self.tmlp = nn.Sequential(nn.Linear(tdim, tdim), nn.SiLU(), nn.Linear(tdim, tdim))
        self.inc = nn.Conv2d(1, ch, 3, padding=1)
        self.down1 = Block(ch, ch, tdim)          # 28x28
        self.down2 = Block(ch, ch * 2, tdim)      # 14x14
        self.mid = Block(ch * 2, ch * 2, tdim)    # 7x7
        self.up2 = Block(ch * 4, ch, tdim)        # 14x14
        self.up1 = Block(ch * 2, ch, tdim)        # 28x28
        self.out = nn.Conv2d(ch, 1, 1)
    def forward(self, x, t):
        temb = self.tmlp(time_embedding(t, self.tdim))
        h1 = self.down1(self.inc(x), temb)
        h2 = self.down2(F.avg_pool2d(h1, 2), temb)
        m = self.mid(F.avg_pool2d(h2, 2), temb)
        u2 = self.up2(torch.cat([F.interpolate(m, scale_factor=2), h2], 1), temb)
        u1 = self.up1(torch.cat([F.interpolate(u2, scale_factor=2), h1], 1), temb)
        return self.out(u1)

model = UNet().to(device)
print(f'parameters: {sum(p.numel() for p in model.parameters()):,}')

## 4. 학습: 섞인 노이즈 맞히기

배치마다 단계 $t$ 를 무작위로 뽑고, 노이즈를 섞은 뒤, 모델이 그 노이즈를 맞히도록 MSE로 학습합니다.

In [ ]:
EPOCHS = 5   # T4 기준 한 에폭에 1~2분 정도 걸립니다. 시간이 있으면 늘려 보세요.
opt = torch.optim.Adam(model.parameters(), lr=2e-4)
losses = []
for epoch in range(EPOCHS):
    for i, (x0, _) in enumerate(loader):
        x0 = x0.to(device)
        t = torch.randint(0, T, (x0.size(0),), device=device)
        noise = torch.randn_like(x0)
        loss = F.mse_loss(model(q_sample(x0, t, noise), t), noise)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    print(f'epoch {epoch + 1}/{EPOCHS}  loss {sum(losses[-100:]) / 100:.4f}')

plt.plot(losses); plt.xlabel('step'); plt.ylabel('MSE loss'); plt.yscale('log'); plt.show()

## 5. 생성: 순수 노이즈에서 숫자로

$x_T\sim\mathrm{N}(0,I)$ 에서 시작해 $t=T-1,\dots,0$ 순서로

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\Big(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\epsilon_\theta(x_t,t)\Big) + \sqrt{\beta_t}\,z$$

를 반복합니다 (마지막 단계에서는 $z=0$).

In [ ]:
@torch.no_grad()
def sample(n=16, keep=(999, 700, 400, 200, 100, 50, 0)):
    model.eval()
    x = torch.randn(n, 1, 28, 28, device=device)
    snapshots = {}
    for t in reversed(range(T)):
        tt = torch.full((n,), t, device=device)
        eps = model(x, tt)
        x = (x - betas[t] / (1 - alpha_bar[t]).sqrt() * eps) / alphas[t].sqrt()
        if t > 0:
            x = x + betas[t].sqrt() * torch.randn_like(x)
        if t in keep:
            snapshots[t] = x.clamp(-1, 1).cpu()
    model.train()
    return x.clamp(-1, 1).cpu(), snapshots

samples, snaps = sample()

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for ax, img in zip(axes.flat, samples):
    ax.imshow(img[0], cmap='gray'); ax.axis('off')
plt.suptitle('generated samples'); plt.show()

In [ ]:
# 노이즈가 숫자로 바뀌어 가는 과정 (위에서 만든 첫 번째 샘플)
ts = sorted(snaps, reverse=True)
fig, axes = plt.subplots(1, len(ts), figsize=(14, 2))
for ax, t in zip(axes, ts):
    ax.imshow(snaps[t][0, 0], cmap='gray'); ax.set_title(f't={t}'); ax.axis('off')
plt.show()

## 생각해 볼 것

- `EPOCHS` 를 1로 줄이거나 20으로 늘리면 생성 품질이 어떻게 달라지나요?
- 생성할 때 `T` 단계를 모두 밟지 않고 건너뛰면 어떻게 될까요? (힌트: DDIM)
- Stable Diffusion은 같은 원리를 **이미지 대신 VAE의 잠재 공간**에서, **텍스트 조건**을 붙여 돌린 것입니다.

## (선택) 같은 원리의 대형 모델: Stable Diffusion 1.5

가중치를 약 4GB 내려받으므로 처음 실행은 몇 분 걸립니다. T4 GPU에서 한 장에 몇 초 정도입니다.

In [ ]:
!pip -q install diffusers transformers accelerate

from diffusers import StableDiffusionPipeline
pipe = StableDiffusionPipeline.from_pretrained(
    'stable-diffusion-v1-5/stable-diffusion-v1-5', torch_dtype=torch.float16).to('cuda')

image = pipe('a watercolor painting of a cat reading a book', num_inference_steps=25, guidance_scale=7.5).images[0]
image